In [1]:
from Bio import SeqIO
from tqdm import tqdm
import re
import numpy as np
import pandas as pd

# parse cutting sites

In [2]:
#site_str = "C|CGG;|AATT"
site_str = "C|CGG;|AATT"

In [3]:
def parse_site(site_str):
    site_str_split= site_str.split(";")
    spot_list = [site.rfind("|") for site in site_str_split]
    site_list = [site.replace("|", "" ).upper() for site in site_str_split]
    site_len_list = [len(site) for site in site_list]

    return [site_list, spot_list, site_len_list]

In [4]:
site_list, spot_list, site_len_list = parse_site(site_str)

In [5]:
site_list

['CCGG', 'AATT']

In [6]:
spot_list

[1, 0]

In [7]:
site_len_list

[4, 4]

In [8]:
site_cuts = {ix:0 for ix in range(len(site_list))}

In [9]:
site_cuts

{0: 0, 1: 0}

# parse genome

In [10]:
ref_fasta = "/home/wbguo/iproject/BSReadSim/test/data/GRCh38_full_analysis_set_plus_decoy_hla.fa"

In [11]:
ref_dict  = SeqIO.to_dict(SeqIO.parse(ref_fasta, "fasta"))

In [12]:
contig_id = 'chr21'

In [13]:
arr_list = []
contig_seq = ref_dict[contig_id].seq.upper()

In [14]:
# get all the cutting sites and arrange
for ix, site in enumerate(site_list):
    arr_list.append(np.array([[match.start(), ix] for match in re.finditer(site, str(contig_seq))], dtype=np.int32))
arr_list.append(np.array([[len(contig_seq), 0]]))

In [15]:
pos_arr = np.concatenate(arr_list, axis=0)
pos_arr = pos_arr[pos_arr[:, 0].argsort()]
pos_num = pos_arr.shape[0]

In [16]:
pos_arr

array([[ 5010065,        1],
       [ 5010443,        1],
       [ 5010555,        1],
       ...,
       [46699365,        0],
       [46699624,        0],
       [46709983,        0]])

In [17]:
pos_num

325571

In [18]:
insert_min = 150
insert_max = 750

In [19]:
# create fragment without cut sites in the middle
ix = 0
cut_dict = site_cuts
frag_arr = np.zeros((pos_num, 5 + len(site_list)), dtype=np.int32) # pos_left, pos_right, left_cut, right_cut, cg_count
for i in range(pos_num):
    cut_l = 0 if i==0 else pos_arr[i-1,1]
    cut_r = pos_arr[i, 1]

    pos_l = 0 if i==0 else pos_arr[i-1,0] + spot_list[cut_l]
    pos_r = pos_arr[i,0] + site_len_list[cut_r] - spot_list[cut_r]

    length= pos_r - pos_l
    if length >= insert_min and length <= insert_max:
        cg_count= (contig_seq[pos_l:pos_r].count('C') + contig_seq[pos_l:pos_r].count('G'))
        frag_arr[ix,:] = [pos_l, pos_r, cut_l, cut_r, cg_count] + list(cut_dict.values())
        ix += 1
frag_arr = frag_arr[:ix,:]
frag_num = frag_arr.shape[0]

In [20]:
frag_arr

array([[ 5010065,  5010447,        1, ...,      196,        0,        0],
       [ 5010555,  5010762,        1, ...,      103,        0,        0],
       [ 5010958,  5011519,        0, ...,      265,        0,        0],
       ...,
       [46697175, 46697380,        1, ...,      106,        0,        0],
       [46697503, 46698134,        0, ...,      279,        0,        0],
       [46699366, 46699627,        0, ...,      158,        0,        0]],
      dtype=int32)

In [21]:
frag_num

85194

In [22]:
# create frag with cut sites in the middle
frag_list = []
for i in range(frag_num):
    pos_l = frag_arr[i,0]
    cut_l = frag_arr[i,2]
    cg_count = frag_arr[i, 4]
    cut_dict = {ix:0 for ix in range(len(site_list))}

    for j in range(i+1, frag_num):
        pos_r= frag_arr[j,1]
        cut_r= frag_arr[j,3]
        if pos_r - pos_l > insert_max:
            break

        cut_dict[frag_arr[j-1,3]] += 1
        cg_count += frag_arr[j, 4]
        frag_list.append([pos_l, pos_r, cut_l, cut_r, cg_count] + list(cut_dict.values()))

In [23]:
frag_arr2= np.concatenate([frag_arr, np.array(frag_list, dtype=np.int32)], axis=0)
frag_arr2= frag_arr2[np.lexsort((frag_arr2[:,1], frag_arr2[:,0]))]

In [24]:
frag_arr2.shape

(150725, 7)

In [25]:
frag_df  = pd.DataFrame(frag_arr2, columns= ['start', 'end', 'cut_l', 'cut_r', 'cg_count'] + site_list)

In [26]:
frag_df.loc[:, 'chr_id'] = contig_id
frag_df.loc[:, 'strand'] = "."
frag_df.loc[:, 'length'] = frag_df.loc[:,'end'] - frag_df.loc[:,'start']
frag_df.loc[:, 'ratio']  = frag_df.loc[:,'cg_count']/frag_df.loc[:,'length']
frag_df.loc[:, 'name']   = "."
#frag_df.loc[:, 'name']   = frag_df.chr_id + ":" + frag_df["start"].astype(str) + "-" + frag_df["end"].astype(str)

In [27]:
frag_df

,start,end,cut_l,cut_r,cg_count,CCGG,AATT,chr_id,strand,length,ratio,name
0,5010065,5010447,1,1,196,0,0,chr21,.,382,0.513089,.
1,5010065,5010762,1,1,299,0,1,chr21,.,697,0.428981,.
2,5010555,5010762,1,1,103,0,0,chr21,.,207,0.497585,.
3,5010958,5011519,0,1,265,0,0,chr21,.,561,0.472371,.
4,5011586,5011969,0,1,248,0,0,chr21,.,383,0.647520,.
...,...,...,...,...,...,...,...,...,...,...,...,...
150720,46696983,46697179,1,1,90,0,0,chr21,.,196,0.459184,.
150721,46696983,46697380,1,1,196,0,1,chr21,.,397,0.493703,.
150722,46697175,46697380,1,1,106,0,0,chr21,.,205,0.517073,.
150723,46697503,46698134,0,1,279,0,0,chr21,.,631,0.442155,.


# check the boundary

In [28]:
contig_seq[frag_df.loc[0, 'start']:frag_df.loc[0, 'end']]

Seq('AATTTTTATTTAATAAGAATGACAGAGTGAGGGCCATCACTGTTAATGAAGCCA...ATT')

In [30]:
frag_df.loc[0, 'end']- frag_df.loc[0, 'start']

382

In [31]:
len(contig_seq[frag_df.loc[0, 'start']:frag_df.loc[0, 'end']])

382

In [ ]:
# for CCGG
contig_seq[(frag_df.loc[0, 'start']-1):(frag_df.loc[0, 'end']+1)]

In [29]:
contig_seq[frag_df.loc[3, 'start']:frag_df.loc[3, 'end']]

Seq('CGGGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGAGCTGAGGAGTTCAAG...ATT')

In [ ]:
frag_df.to_csv('~/test.txt', columns=['chr_id', 'start', 'end', 'name', 'strand'], sep = '\t', mode='w', header=False, index=False, quoting=None)